In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
import numpy as np

# Define file paths
data_path = "/kaggle/input/brain-fmri-dataset/all_segmented_data.npy"
labels_path = "/kaggle/input/brain-fmri-dataset/all_labels.npy"

# Load the data
all_segmented_data = np.load(data_path)
all_labels = np.load(labels_path)

# Print the shapes
print("Shape of all_segmented_data:", all_segmented_data.shape)
print("Shape of all_labels:", all_labels.shape)


In [ ]:
print(all_labels)

In [ ]:
import matplotlib.pyplot as plt
import random

def plot_sample_slices(sample, label, sample_index):
    """
    Visualize one sample as a 4x9 grid.
    - sample: a 3D array of shape (100, 100, 36)
    - label: corresponding label for the sample (string)
    - sample_index: index of the sample (for display purposes)
    """
    # Create a grid of subplots: 4 rows (time points) x 9 columns (slices)
    fig, axes = plt.subplots(4, 9, figsize=(12, 6))
    fig.suptitle(f"Sample Index: {sample_index} | Label: {label}", fontsize=16)
    
    # Loop over time points (rows) and slices (columns)
    for t in range(4):           # 4 time points
        for s in range(9):       # 9 slices per time point
            ax = axes[t, s]
            # Calculate the channel index for the current slice:
            channel_idx = t * 9 + s
            # Display the slice (as grayscale)
            ax.imshow(sample[:, :, channel_idx], cmap='gray')
            ax.axis('off')
    
    plt.tight_layout(rect=[0, 0.03, 1, 0.95])
    plt.show()

# Choose a few random sample indices to visualize
num_samples_to_visualize = 2  # Adjust the number of samples you'd like to check
indices = random.sample(range(all_segmented_data.shape[0]), num_samples_to_visualize)

# Visualize each chosen sample
for idx in indices:
    plot_sample_slices(all_segmented_data[idx], all_labels[idx], idx)


In [ ]:
import numpy as np
from sklearn.model_selection import train_test_split
from tensorflow.keras.utils import to_categorical

Step 2: Preprocess the Labels

In [ ]:
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.utils import to_categorical

# Convert labels to strings (if not already) and then encode them as integers
all_labels = all_labels.astype(str)
label_encoder = LabelEncoder()
labels_encoded = label_encoder.fit_transform(all_labels)

# Get number of unique classes and convert labels to one-hot vectors
num_classes = len(label_encoder.classes_)
y = to_categorical(labels_encoded, num_classes=num_classes)

print("Classes found:", label_encoder.classes_)
print("One-hot labels shape:", y.shape)


In [ ]:
y

Normalize

In [ ]:
# Convert the data type to float32
X = all_segmented_data.astype('float32')

# Normalize the data: subtract the minimum and divide by the range
X_min = X.min()
X_max = X.max()
X = (X - X_min) / (X_max - X_min)

print("Data normalized. New min:", X.min(), "New max:", X.max())


**Split the Data into Training, Validation, and Test Sets**

In [ ]:
from sklearn.model_selection import train_test_split

# First, split into training (70%) and temporary (30% for validation and test)
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.3, random_state=42)

# Now, split the temporary set into validation and test sets (each 50% of 30% i.e., 15% of total)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42)

print("Training set shape:", X_train.shape, y_train.shape)
print("Validation set shape:", X_val.shape, y_val.shape)
print("Test set shape:", X_test.shape, y_test.shape)


**CNN Model**

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout

model = Sequential([
    # First convolutional block
    Conv2D(32, (3, 3), activation='relu', input_shape=(100, 100, 36)),
    MaxPooling2D((2, 2)),
    
    # Second convolutional block
    Conv2D(64, (3, 3), activation='relu'),
    MaxPooling2D((2, 2)),
    
    # Flatten the feature maps
    Flatten(),
    
    # Fully connected layer with dropout for regularization
    Dense(128, activation='relu'),
    Dropout(0.5),
    
    # Output layer with softmax activation for multi-class classification
    Dense(num_classes, activation='softmax')
])

# Display the model architecture
model.summary()


**Compile and Train the Model**

In [ ]:
# Compile the model
model.compile(optimizer='adam',
              loss='categorical_crossentropy',
              metrics=['accuracy'])

# Train the model
history = model.fit(X_train, y_train,
                    epochs=20,           # Adjust epochs as needed
                    batch_size=32,       # Adjust batch size as needed
                    validation_data=(X_val, y_val))


In [ ]:
test_loss, test_accuracy = model.evaluate(X_test, y_test)
print("Test accuracy:", test_accuracy)


In [ ]:
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.utils import to_categorical

all_labels = all_labels.astype(str)
label_encoder = LabelEncoder()
labels_encoded = label_encoder.fit_transform(all_labels)

num_classes = len(label_encoder.classes_)
y = to_categorical(labels_encoded, num_classes=num_classes)

print("Unique classes:", label_encoder.classes_)
print("Shape of one-hot labels:", y.shape)


In [ ]:
X = all_segmented_data.astype('float32')

X_min = X.min()
X_max = X.max()
X = (X - X_min) / (X_max - X_min)

print("Data shape:", X.shape)
print("Data range after normalization:", X.min(), X.max())


In [ ]:
from sklearn.model_selection import train_test_split

# 70% train, 30% temp
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.3, random_state=42
)

# Of the 30% temp, half goes to validation, half to test
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42
)

print("Training set shape:", X_train.shape, y_train.shape)
print("Validation set shape:", X_val.shape, y_val.shape)
print("Test set shape:", X_test.shape, y_test.shape)


In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout, BatchNormalization

model = Sequential()

# Block 1
model.add(Conv2D(64, (3,3), padding='same', activation='relu', input_shape=(100, 100, 36)))
model.add(BatchNormalization())
model.add(Conv2D(64, (3,3), padding='same', activation='relu'))
model.add(BatchNormalization())
model.add(MaxPooling2D((2,2)))
model.add(Dropout(0.25))

# Block 2
model.add(Conv2D(128, (3,3), padding='same', activation='relu'))
model.add(BatchNormalization())
model.add(Conv2D(128, (3,3), padding='same', activation='relu'))
model.add(BatchNormalization())
model.add(MaxPooling2D((2,2)))
model.add(Dropout(0.25))

# Block 3
model.add(Conv2D(256, (3,3), padding='same', activation='relu'))
model.add(BatchNormalization())
model.add(Conv2D(256, (3,3), padding='same', activation='relu'))
model.add(BatchNormalization())
model.add(MaxPooling2D((2,2)))
model.add(Dropout(0.25))

# Classification head
model.add(Flatten())
model.add(Dense(512, activation='relu'))
model.add(BatchNormalization())
model.add(Dropout(0.5))
model.add(Dense(num_classes, activation='softmax'))

model.summary()


In [ ]:
model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)


In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

datagen = ImageDataGenerator(
    width_shift_range=0.1,
    height_shift_range=0.1,
    rotation_range=10,
    horizontal_flip=True,
    zoom_range=0.1
)
datagen.fit(X_train)  # Compute any statistics required for augmentation


In [ ]:
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint



history = model.fit(
    X_train, y_train,
    epochs=30,
    batch_size=32,
    validation_data=(X_val, y_val),
)


In [ ]:
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv3D, MaxPooling3D, Flatten, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

# ------------------------------------------------------------------------------
# 1. Load the Data
# ------------------------------------------------------------------------------

all_segmented_data = np.load(data_path)  # shape: (1280, 100, 100, 36)
all_labels = np.load(labels_path)        # e.g. ['four' 'child' 'father' ... ]

# ------------------------------------------------------------------------------
# 2. Preprocess the Labels (string -> int -> one-hot)
# ------------------------------------------------------------------------------
all_labels = all_labels.astype(str)
label_encoder = LabelEncoder()
labels_encoded = label_encoder.fit_transform(all_labels)
num_classes = len(label_encoder.classes_)
y = to_categorical(labels_encoded, num_classes=num_classes)

print("Unique classes:", label_encoder.classes_)
print("Labels one-hot shape:", y.shape)

# ------------------------------------------------------------------------------
# 3. Preprocess the Data (Normalize + Reshape for 3D CNN)
# ------------------------------------------------------------------------------
X = all_segmented_data.astype('float32')

# Normalize to [0, 1] (or any other normalization you prefer)
X_min = X.min()
X_max = X.max()
X = (X - X_min) / (X_max - X_min)

print("Original X shape:", X.shape)  # (1280, 100, 100, 36)

# Reshape into 3D volumes with shape: (depth=36, height=100, width=100, channels=1)
# Keras Conv3D expects input_shape=(depth, height, width, channels) when data_format='channels_last'.
X_3d = X.reshape((X.shape[0], 36, 100, 100, 1))
print("Reshaped X for 3D CNN:", X_3d.shape)  # (1280, 36, 100, 100, 1)

# ------------------------------------------------------------------------------
# 4. Split Into Training, Validation, and Test Sets
# ------------------------------------------------------------------------------
X_train, X_temp, y_train, y_temp = train_test_split(X_3d, y, test_size=0.3, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42)

print("Training set shape:", X_train.shape, y_train.shape)
print("Validation set shape:", X_val.shape, y_val.shape)
print("Test set shape:", X_test.shape, y_test.shape)

# ------------------------------------------------------------------------------
# 5. Define a 3D CNN Model
# ------------------------------------------------------------------------------
model = Sequential()

# 3D convolution block 1
model.add(Conv3D(32, kernel_size=(3, 3, 3), activation='relu', padding='same',
                 input_shape=(36, 100, 100, 1)))  # depth=36, height=100, width=100, channel=1
model.add(MaxPooling3D(pool_size=(2, 2, 2)))  # pool in depth, height, width

# 3D convolution block 2
model.add(Conv3D(64, kernel_size=(3, 3, 3), activation='relu', padding='same'))
model.add(MaxPooling3D(pool_size=(2, 2, 2)))

# 3D convolution block 3 (optional deeper block)
model.add(Conv3D(128, kernel_size=(3, 3, 3), activation='relu', padding='same'))
model.add(MaxPooling3D(pool_size=(2, 2, 2)))

model.add(Flatten())
model.add(Dense(256, activation='relu'))
model.add(Dropout(0.5))
model.add(Dense(num_classes, activation='softmax'))

model.summary()

# ------------------------------------------------------------------------------
# 6. Compile the Model
# ------------------------------------------------------------------------------
model.compile(optimizer='adam',
              loss='categorical_crossentropy',
              metrics=['accuracy'])

# ------------------------------------------------------------------------------
# 7. Train the Model with Callbacks (EarlyStopping + ModelCheckpoint)
# ------------------------------------------------------------------------------
early_stop = EarlyStopping(patience=5, restore_best_weights=True)

# If you want to save the entire model, use .keras
checkpointer = ModelCheckpoint(
    "best_model.keras",
    monitor='val_accuracy',
    save_best_only=True,
    verbose=1
)

history = model.fit(
    X_train, y_train,
    epochs=30,
    batch_size=4,  # smaller batch may help with memory for 3D data
    validation_data=(X_val, y_val),
    callbacks=[early_stop, checkpointer]
)

# ------------------------------------------------------------------------------
# 8. Evaluate the Model on the Test Set
# ------------------------------------------------------------------------------
test_loss, test_accuracy = model.evaluate(X_test, y_test)
print("Test accuracy:", test_accuracy)

# ------------------------------------------------------------------------------
# 9. (Optional) Load the Best Model
# ------------------------------------------------------------------------------
from tensorflow.keras.models import load_model
best_model = load_model("best_model.keras")
best_test_loss, best_test_accuracy = best_model.evaluate(X_test, y_test)
print("Best model test accuracy:", best_test_accuracy)


**------**

In [ ]:
import numpy as np

# Define file paths
data_path = "/kaggle/input/brain-fmri-dataset/all_segmented_data.npy"
labels_path = "/kaggle/input/brain-fmri-dataset/all_labels.npy"

# Load the data
all_segmented_data = np.load(data_path)
all_labels = np.load(labels_path)

# Print the shapes
print("Shape of all_segmented_data:", all_segmented_data.shape)
print("Shape of all_labels:", all_labels.shape)
       # e.g. ['four' 'child' 'father' ... ]


In [ ]:
import numpy as np
import pandas as pd

# Check for NaN values in all_segmented_data (assuming it is a numeric array)
nan_data = np.isnan(all_segmented_data)
nan_count_data = np.sum(nan_data)

# Check for NaN or None values in all_labels (assuming it is an array of strings or categorical labels)
# If all_labels are strings or categorical, we use pd.isnull
nan_labels = pd.isnull(all_labels)
nan_count_labels = np.sum(nan_labels)

# Print the counts
print("Number of NaN values in all_segmented_data:", nan_count_data)
print("Number of NaN values in all_labels:", nan_count_labels)


In [ ]:
all_labels

In [ ]:
from sklearn.preprocessing import LabelEncoder

# Initialize the label encoder
label_encoder = LabelEncoder()

# Fit and transform the labels
encoded_labels = label_encoder.fit_transform(all_labels)


In [ ]:
encoded_labels

In [ ]:
import numpy as np
from sklearn.model_selection import train_test_split
from keras.models import Sequential
from keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Activation
from keras.optimizers import Adam
from keras.callbacks import EarlyStopping
from keras.utils import to_categorical

# Determine number of classes from encoded_labels
n_classes = len(np.unique(encoded_labels))
print("Number of classes:", n_classes)

# Split the data into training and testing sets (adjust test_size if needed)
X_train, X_test, y_train, y_test = train_test_split(
    all_segmented_data, encoded_labels, test_size=0.2, random_state=42
)

# Convert labels to one-hot encoding
y_train_cat = to_categorical(y_train, num_classes=n_classes)
y_test_cat = to_categorical(y_test, num_classes=n_classes)

# Define the CNN model using the architecture from the GitHub notebook
def make_custom_model_cnn_2D(input_shape, n_classes):
    model = Sequential()
    
    # Convolutional block 1
    model.add(Conv2D(8, (3,3), kernel_initializer='he_normal', padding='same', input_shape=input_shape))
    model.add(Activation('relu'))
    model.add(MaxPooling2D(pool_size=(2,2)))
    
    # Convolutional block 2
    model.add(Conv2D(16, (3,3), kernel_initializer='he_normal', padding='same'))
    model.add(Activation('relu'))
    model.add(MaxPooling2D(pool_size=(2,2)))
    
    # Convolutional block 3
    model.add(Conv2D(32, (3,3), kernel_initializer='he_normal', padding='same'))
    model.add(Activation('relu'))
    model.add(MaxPooling2D(pool_size=(2,2)))
    
    # Convolutional block 4
    model.add(Conv2D(64, (3,3), kernel_initializer='he_normal', padding='same'))
    model.add(Activation('relu'))
    model.add(MaxPooling2D(pool_size=(2,2)))
    
    # Transition to fully connected layers
    model.add(Flatten()) 
    model.add(Dense(128, kernel_initializer='he_normal'))
    model.add(Activation('relu'))
    
    # Output layer: note that we set units to n_classes
    model.add(Dense(n_classes, kernel_initializer='he_normal'))
    model.add(Activation('linear'))
    model.add(Activation('softmax'))
    
    # Compile the model using Adam optimizer
    adam = Adam(learning_rate =0.001)
    model.compile(loss='categorical_crossentropy', optimizer=adam, metrics=['accuracy'])
    
    return model

# Get input shape (should be (100, 100, 36) for your dataset)
input_shape = X_train.shape[1:]
print("Input shape:", input_shape)

# Build the model
model = make_custom_model_cnn_2D(input_shape, n_classes)
model.summary()

# Set up early stopping callback (adjust patience as needed)
early_stop = EarlyStopping(monitor='val_loss', patience=5, verbose=1)

# Train the model (adjust epochs and batch_size as needed)
history = model.fit(
    X_train, y_train_cat,
    batch_size=128,
    epochs=50,
    validation_data=(X_test, y_test_cat),
    # callbacks=[early_stop],
    shuffle=True,
    verbose=1
)

# Evaluate the model on the test set
score = model.evaluate(X_test, y_test_cat, verbose=0)
print("\nTest Accuracy: {:.4f}".format(score[1]))


In [ ]:
import numpy as np
from sklearn.model_selection import train_test_split
from keras.models import Sequential
from keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Activation
from keras.optimizers import Adam
from keras.callbacks import EarlyStopping
from keras.utils import to_categorical
from keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Activation, BatchNormalization, Dropout

# Example normalization: scale data to [0, 1]
all_segmented_data = all_segmented_data.astype('float32') / 255.0

# Determine number of classes from encoded_labels
n_classes = len(np.unique(encoded_labels))
print("Number of classes:", n_classes)

# Split the data into training and testing sets (adjust test_size if needed)
X_train, X_test, y_train, y_test = train_test_split(
    all_segmented_data, encoded_labels, test_size=0.2, random_state=42
)

# Convert labels to one-hot encoding
y_train_cat = to_categorical(y_train, num_classes=n_classes)
y_test_cat = to_categorical(y_test, num_classes=n_classes)

# Define the CNN model using the architecture from the GitHub notebook
def make_custom_model_cnn_2D(input_shape, n_classes):
    model = Sequential()
    
    # Convolutional block 1
    model.add(Conv2D(8, (3,3), kernel_initializer='he_normal', padding='same', input_shape=input_shape))
    model.add(BatchNormalization())
    model.add(Activation('relu'))
    model.add(MaxPooling2D(pool_size=(2,2)))
    model.add(Dropout(0.25))
    
    # Convolutional block 2
    model.add(Conv2D(16, (3,3), kernel_initializer='he_normal', padding='same'))
    model.add(BatchNormalization())
    model.add(Activation('relu'))
    model.add(MaxPooling2D(pool_size=(2,2)))
    model.add(Dropout(0.25))
    
    # Convolutional block 3
    model.add(Conv2D(32, (3,3), kernel_initializer='he_normal', padding='same'))
    model.add(BatchNormalization())
    model.add(Activation('relu'))
    model.add(MaxPooling2D(pool_size=(2,2)))
    model.add(Dropout(0.25))
    
    # Convolutional block 4
    model.add(Conv2D(64, (3,3), kernel_initializer='he_normal', padding='same'))
    model.add(BatchNormalization())
    model.add(Activation('relu'))
    model.add(MaxPooling2D(pool_size=(2,2)))
    model.add(Dropout(0.25))
    
    model.add(Flatten())
    model.add(Dense(128, kernel_initializer='he_normal'))
    model.add(Activation('relu'))
    model.add(Dropout(0.5))
    
    model.add(Dense(n_classes, kernel_initializer='he_normal'))
    model.add(Activation('softmax'))
    
    # Compile the model using the updated Adam optimizer parameter
    adam = Adam(learning_rate=0.001)
    model.compile(loss='categorical_crossentropy', optimizer=adam, metrics=['accuracy'])
    
    return model

# Get input shape (should be (100, 100, 36) for your dataset)
input_shape = X_train.shape[1:]
print("Input shape:", input_shape)

# Build the model
model = make_custom_model_cnn_2D(input_shape, n_classes)
model.summary()

# Set up early stopping callback (adjust patience as needed)
early_stop = EarlyStopping(monitor='val_loss', patience=5, verbose=1)

# Train the model (adjust epochs and batch_size as needed)
history = model.fit(
    X_train, y_train_cat,
    batch_size=128,
    epochs=50,
    validation_data=(X_test, y_test_cat),
    # callbacks=[early_stop],
    shuffle=True,
    verbose=1
)

# Evaluate the model on the test set
score = model.evaluate(X_test, y_test_cat, verbose=0)
print("\nTest Accuracy: {:.4f}".format(score[1]))


In [ ]:
import torch
import torch.nn as nn
from torchvision.models.video import swin_transformer

# Define the model
class fMRISwinTransformer(nn.Module):
    def __init__(self, num_classes):
        super(fMRISwinTransformer, self).__init__()
        # Initialize the Swin Transformer 3D model
        self.swin_transformer = swin_transformer.SwinTransformer3d(
            embed_dim=96,
            depths=(2, 2, 6, 2),
            num_heads=(3, 6, 12, 24),
            window_size=(2, 7, 7),
            patch_size=(2, 4, 4),
            num_classes=num_classes
        )

    def forward(self, x):
        x = self.swin_transformer(x)
        return x

# Initialize the model
num_classes = len(set(all_labels))
model = fMRISwinTransformer(num_classes=num_classes)


In [ ]:
# Add a channel dimension to the data
all_segmented_data = np.expand_dims(all_segmented_data, axis=1)

# Convert the data to torch tensors
X = torch.tensor(all_segmented_data, dtype=torch.float32)
y = torch.tensor(encoded_labels, dtype=torch.long)


In [ ]:
# Check the first few elements
print(encoded_labels[:5])
print(y[:5])


In [ ]:
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, TensorDataset

# Split the data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Create data loaders
train_dataset = TensorDataset(X_train, y_train)
test_dataset = TensorDataset(X_test, y_test)
train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=8, shuffle=False)

# Define loss function and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)


In [ ]:
print("Shape of all_segmented_data:", all_segmented_data.shape)
print("Shape of all_labels:", all_labels.shape)

In [ ]:
import numpy as np
import torch
from sklearn.preprocessing import LabelEncoder

# Define file paths
data_path = "/kaggle/input/brain-fmri-dataset/all_segmented_data.npy"
labels_path = "/kaggle/input/brain-fmri-dataset/all_labels.npy"

# Load the data
all_segmented_data = np.load(data_path)
all_labels = np.load(labels_path)

print("Shape of all_segmented_data:", all_segmented_data.shape)
print("Shape of all_labels:", all_labels.shape)

# --- Preprocessing Steps ---

# 1. Add a channel dimension (from [B, H, W, D] to [B, C, H, W, D])
all_segmented_data = np.expand_dims(all_segmented_data, axis=1)  # Now shape: (1280, 1, 100, 100, 36)

# 2. Permute dimensions to have the shape [B, C, T, H, W]
# Here, we assume the last dimension (36) is the temporal dimension.
all_segmented_data = np.transpose(all_segmented_data, (0, 1, 4, 2, 3))  # Now shape: (1280, 1, 36, 100, 100)

# 3. Repeat the channel dimension to match the model's expected input channels (3 channels)
all_segmented_data = np.repeat(all_segmented_data, 3, axis=1)  # Now shape: (1280, 3, 36, 100, 100)

# 4. Convert the data to PyTorch tensors
X = torch.tensor(all_segmented_data, dtype=torch.float32)

# 5. Encode the string labels into integers
label_encoder = LabelEncoder()
encoded_labels = label_encoder.fit_transform(all_labels)
y = torch.tensor(encoded_labels, dtype=torch.long)

# Verify the new shape and labels
print("New shape of X:", X.shape)  # Expected: (1280, 3, 36, 100, 100)
print("First 5 encoded labels:", y[:5])


In [ ]:

import torch
import torch.nn as nn
from torchvision.models.video import swin_transformer

class fMRISwinTransformer(nn.Module):
    def __init__(self, num_classes):
        super(fMRISwinTransformer, self).__init__()
        # Pass window_size and patch_size as lists rather than tuples
        self.swin_transformer = swin_transformer.SwinTransformer3d(
            embed_dim=96,
            depths=(2, 2, 6, 2),
            num_heads=(3, 6, 12, 24),
            window_size=[2, 7, 7],   # Changed from tuple to list
            patch_size=[2, 4, 4],    # Changed from tuple to list
            num_classes=num_classes
        )

    def forward(self, x):
        x = self.swin_transformer(x)
        return x

# Example instantiation:
# Ensure you have already encoded your labels into integers
num_classes = len(set(encoded_labels))  # or use your precomputed number of classes
model = fMRISwinTransformer(num_classes=num_classes)


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split

# --- Assume X and y are already prepared ---
# X: Tensor of shape [1280, 3, 36, 100, 100] (your fMRI data)
# y: Tensor of shape [1280] (encoded labels)

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Create TensorDatasets and DataLoaders
train_dataset = TensorDataset(X_train, y_train)
test_dataset  = TensorDataset(X_test, y_test)
train_loader  = DataLoader(train_dataset, batch_size=8, shuffle=True)
test_loader   = DataLoader(test_dataset, batch_size=8, shuffle=False)

# Define loss function and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# --- Training Loop ---
num_epochs = 10
for epoch in range(num_epochs):
    model.train()  # Set model to training mode
    running_loss = 0.0
    for inputs, labels in train_loader:
        optimizer.zero_grad()          # Zero the gradients
        outputs = model(inputs)        # Forward pass
        loss = criterion(outputs, labels)  # Compute loss
        loss.backward()                # Backpropagation
        optimizer.step()               # Update weights
        running_loss += loss.item() * inputs.size(0)
    
    epoch_loss = running_loss / len(train_loader.dataset)
    print(f"Epoch {epoch+1}/{num_epochs}, Loss: {epoch_loss:.4f}")

# --- Evaluation on Test Set ---
model.eval()  # Set model to evaluation mode
all_preds = []
all_true = []

with torch.no_grad():
    for inputs, labels in test_loader:
        outputs = model(inputs)
        _, preds = torch.max(outputs, 1)
        all_preds.extend(preds.cpu().numpy())
        all_true.extend(labels.cpu().numpy())

from sklearn.metrics import accuracy_score
accuracy = accuracy_score(all_true, all_preds)
print(f"Test Accuracy: {accuracy:.4f}")
